# 오류와 읽기 도구·쓰기 도구 분리

오류 코드와 제공 경로를 분리합니다.

## 준비

In [1]:
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = Path("data/orders.csv")
assert data_path.exists(), "노트북은 labs/part5/ch19-model-tools에서 실행합니다."
print("Python 실습 준비 완료")

Python 실습 준비 완료


In [2]:
from langchain.tools import tool
ORDERS = __import__("pandas").read_csv(data_path).to_dict(orient="records")

## 읽기 도구

In [3]:
@tool
def read_order(order_id: str) -> dict:
    """주문 ID로 조회하며 데이터를 변경하지 않습니다."""
    row = next((r for r in ORDERS if r["order_id"] == order_id), None)
    return {"ok": True, "order": row} if row else {"ok": False, "error": "NOT_FOUND", "order_id": order_id}
print(read_order.invoke({"order_id": "O999"}))

{'ok': False, 'error': 'NOT_FOUND', 'order_id': 'O999'}


## 쓰기 도구

In [4]:
@tool
def request_order_cancellation(order_id: str) -> dict:
    """애플리케이션이 확인한 주문의 취소 요청을 기록합니다."""
    return {"ok": True, "status": "cancellation_requested", "order_id": order_id}

## 확인 전

In [5]:
app_user_confirmed = False  # 신뢰하는 애플리케이션 상태의 실습값
available_tools = [read_order]
print({"confirmed": app_user_confirmed, "tools": [t.name for t in available_tools]})
assert request_order_cancellation not in available_tools

{'confirmed': False, 'tools': ['read_order']}


## 애플리케이션이 사용자 확인을 기록한 뒤

In [6]:
app_user_confirmed = True  # 실제 서비스에서는 UI 확인 결과를 서버가 기록합니다.
available_tools = [read_order, request_order_cancellation] if app_user_confirmed else [read_order]
print({"confirmed": app_user_confirmed, "tools": [t.name for t in available_tools]})
result = request_order_cancellation.invoke({"order_id": "O101"})
print(result)

{'confirmed': True, 'tools': ['read_order', 'request_order_cancellation']}
{'ok': True, 'status': 'cancellation_requested', 'order_id': 'O101'}
